<a href="https://colab.research.google.com/github/marian-bosibor/Special_Topics_Lab-1/blob/main/Network_Automation_Marian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install --upgrade pip
!pip install requests netmiko napalm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 535.8/535.8 kB 9.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.4/642.4 kB 16.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 47.3 MB/s  0:00:00
  Created wheel for ncclient: filename=ncclient-0.7.0-py3-none-any.whl size=94121 sha256=b6bada7ba0ddf040786e756ae9cc860395f5695dadd0209bfb58fa057767e5bb
  Stored in directory: /root/.cache/pip/wheels/7a/4b/6

In [7]:
!pip install netmiko

In [6]:
!pip install --upgrade netmiko

In [8]:
!pip show netmiko

Name: netmiko
Version: 4.6.0
Summary: Multi-vendor library to simplify legacy CLI connections to network devices
Home-page: https://github.com/ktbyers/netmiko
Author: Kirk Byers
Author-email: ktbyers@twb-tech.com
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: ntc-templates, paramiko, pyserial, pyyaml, rich, ruamel.yaml, scp, textfsm
Required-by: napalm



**Define the Device Credentials**

We use a Cisco DevNet Always-On Sandbox. This is a real router that is always reachable over the internet

ConnectHandler acts as a universal adapter

**Roles of ConnectHandler**

Establishes SSH connection to network device
Automatically handles the handshake (things like logging in, handles the # or > prompts.

**NB **The moment you specify a device type e.g., cisco_ios, ConnectHandler knows how to:

  .specifically navigate that specific OS
  
  .Enter configuration mode

  .Wait for the device to finish processing a command

In [10]:
from netmiko import ConnectHandler

import getpass

 # Device details for your active Cisco DevNet Sandbox
device = {
     'device_type': 'cisco_xe',
     'host':'devnetsandboxiosxec9k.cisco.com',
     'username': 'marian.onyiego',
     'password':'-G4_OkMYkwg60D',
     'port': 22,



 }

print(f"Targeting: {device['host']}")

Targeting: devnetsandboxiosxec9k.cisco.com


In [11]:
try:
   print("Connecting to the device...")
    # Print device details for debugging
   net_connect = ConnectHandler(**device)

   print("\n--- Interface Status ---")
   output = net_connect.send_command("show ip interface brief")
   print(output)

   net_connect.disconnect()
   print("\nDisconnected successfully.")


except Exception as e:
  print(f"Error connecting to the device: {e}")


Connecting to the device...

--- Interface Status ---
Interface              IP-Address      OK? Method Status                Protocol
Vlan1                  unassigned      YES NVRAM  up                    down    
Vlan10                 unassigned      YES unset  administratively down down    
GigabitEthernet0/0     10.10.20.66     YES NVRAM  up                    up      
GigabitEthernet1/0/1   unassigned      YES unset  up                    up      
GigabitEthernet1/0/2   unassigned      YES unset  administratively down down    
GigabitEthernet1/0/3   unassigned      YES unset  administratively down down    
GigabitEthernet1/0/4   unassigned      YES unset  administratively down down    
GigabitEthernet1/0/5   unassigned      YES unset  administratively down down    
GigabitEthernet1/0/6   unassigned      YES unset  administratively down down    
GigabitEthernet1/0/7   unassigned      YES unset  administratively down down    
GigabitEthernet1/0/8   unassigned      YES unset  admin

In [12]:
# Re-establish connections for the config change
# Redefine device with getpass for passwd to ensure correct authentication

device = {
    'device_type' : 'cisco_xe',
    'host': 'devnetsandboxiosxec9k.cisco.com',
    'username': 'marian.onyiego',
    'password': getpass.getpass("Enter router password: "),
    'port': 22,

}

net_connect = ConnectHandler(**device)

config_commands = [
    'interface Loopback99',
    'description configured via python Netmiko',
    'ip address 192.168.99.1 255.255.255.255'
]

print("Sending configurations commands...")
output = net_connect.send_config_set(config_commands)
print(output)

#verify the change
verification = net_connect.send_command("show ip interface brief | include Loopback99")
print("\nVerification")
print(verification)

net_connect.disconnect

Enter router password: ··········
Sending configurations commands...
configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
Cat9k_AO_Sandbox(config)#interface Loopback99
Cat9k_AO_Sandbox(config-if)#description configured via python Netmiko
Cat9k_AO_Sandbox(config-if)#ip address 192.168.99.1 255.255.255.255
Cat9k_AO_Sandbox(config-if)#end
Cat9k_AO_Sandbox#

Verification
Loopback99             192.168.99.1    YES manual up                    up      


<bound method BaseConnection.disconnect of <netmiko.cisco.cisco_ios.CiscoIosSSH object at 0x7fc58450f8f0>>

### Task: Apply a simple configuration such as a loopback to your switch

In [15]:
# Re-establishing the connection
print("Reconnecting to the device...")
net_connect = ConnectHandler(**device)

config_commands = [
    'interface Loopback100',
    'description configured via python Netmiko',
    'ip address 192.168.100.1 255.255.255.255',
]

try:
  print("Sending configurations commands...")
  output = net_connect.send_config_set(config_commands)
  output_2 = net_connect.send_config_set("Show ip interface brief")
  print(output)
  print(output_2)

    # verify the change
  verification = net_connect.send_command("show ip interface brief | include Loopback100")
  print("\nVerification")
  print(verification)

finally:
    net_connect.disconnect()#always close the door when finished
    print("\nDisconnected successfully.")


Reconnecting to the device...
Sending configurations commands...
configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
Cat9k_AO_Sandbox(config)#interface Loopback100
Cat9k_AO_Sandbox(config-if)#description configured via python Netmiko
Cat9k_AO_Sandbox(config-if)#ip address 192.168.100.1 255.255.255.255
Cat9k_AO_Sandbox(config-if)#end
Cat9k_AO_Sandbox#
configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
Cat9k_AO_Sandbox(config)#Show ip interface brief
                           ^
% Invalid input detected at '^' marker.

Cat9k_AO_Sandbox(config)#end
Cat9k_AO_Sandbox#

Verification
Loopback100            192.168.100.1   YES manual up                    up      

Disconnected successfully.


**Automating VLAN Subinterfaces**
We want to see how to create logical subinterfaces that tag traffic with specific VLAN IDs (802.1Q)

In [16]:
#Define VLAN-to-subinterface mapping
vlans_to_config = [
    {'id': 10, 'ip': '192.168.10.1', 'desc': 'SCES_VLAN'},
    {'id': 20, 'ip': '192.168.20.1', 'desc': 'SCES_VLAN'},

]

with ConnectHandler(**device) as net_connect:
    for vlan in vlans_to_config:
      print(f"Configuring VLAN {vlan['id']}")
      config_commands = [
          f"vlan {vlan['id']}",
          f"name {vlan['desc']}",
          f"interface Vlan{vlan['desc']}",
          f"description{vlan['desc']}"
          f"ip address {vlan['ip']} 255.255.255.255",
          "no shutdown"
      ]
      output = net_connect.send_config_set(config_commands)
      print(output)

      #verifications
      print("\nVerifying subinterfaces")
      verify = net_connect.send_command("show vlan brief")
      print(verify)

#

Configuring VLAN 10
configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
Cat9k_AO_Sandbox(config)#vlan 10
Cat9k_AO_Sandbox(config-vlan)#name SCES_VLAN
Cat9k_AO_Sandbox(config-vlan)#interface VlanSCES_VLAN
                                            ^
% Invalid input detected at '^' marker.

Cat9k_AO_Sandbox(config)#descriptionSCES_VLANip address 192.168.10.1 255.255.255.255
                           ^
% Invalid input detected at '^' marker.

Cat9k_AO_Sandbox(config)#no shutdown
% Incomplete command.

Cat9k_AO_Sandbox(config)#end
Cat9k_AO_Sandbox#

Verifying subinterfaces

VLAN Name                             Status    Ports
---- -------------------------------- --------- -------------------------------
1    default                          active    Gi1/0/2, Gi1/0/3, Gi1/0/4, Gi1/0/5, Gi1/0/6, Gi1/0/7, Gi1/0/8
10   SCES_VLAN                        active    Gi1/0/1
90   JIRA-VLAN-90                     active    
99   ANSIBLE-DEMO-VLAN                activ

**Multi-Device Configuration**

We want to try automating multiple virtual instances at once

**Workflow:** Store your device list in a devices.json or yaml

**Lab Task:** We want to write a Python script that loops through the file, connects to each device and pushed a standard "Golden COnfig" (e.g., setting up the same local user, banner, SSH timeout settings, etc)

**JSON**

JavaScript Object Notation (JSON) is a standard format for storing and exchanging data using human-readable text.

We can create JSON scripts/ files for this lab inventory in either of these two methods:

1. Manually in a text editor such as Notepad++ ,TextEdit, or VS Code or
2. Generate using a Python Script as shown in the next cell wehere we create device inventory using JSON.

In [17]:
import json
import getpass

# Define our device using a list of dictionaries

device_list = [
    {
    'device_type' : 'cisco_xe',
    'host': 'devnetsandboxiosxec9k.cisco.com',
    'username': 'marian.onyiego',
    'password': getpass.getpass("Enter router password: "),
    'port': 22,
    },
    {
    'device_type' : 'cisco_xe',
    'host': "192.0.2.10",
    'username': 'developer',
    'password': getpass.getpass("Enter router password: "),
    }
]

# Write the data to a file named 'devicesPY.json'
with open('devicesPY.json', 'w') as f:
    json.dump(device_list, f, indent=4)

print ("devices.json has been created")


Enter router password: ··········
Enter router password: ··········
devices.json has been created



**Multi-Device Automation Script**

The script that follows will load a JSON file, connect to each device in sequence, and pushes a set of configuration commands.

In [20]:
from netmiko import NetmikoTimeoutException, NetmikoAuthenticationException, ConnectHandler
import json # Added missing import

# Load the device fleet from the JSON file

def load_devices(filepath):
  with open(filepath) as f:
   return json.load(f)

#Define the configuration to be applied

config_commands_multiple = [
    'interface GigabitEthernet1/0/1',
    'no switchport',
    'no shutdown',
    'interface GigabitEthernet1/0/1.50',
    'encapsulation dot1Q 50',
   'ip address 192.168.50.1 255.255.255.0',
    'description Automation_VLAN_50',
]

def run_automation():
  devices = load_devices('/content/devicesPY.json') # Using the hardcoded path for consistency

  for device in devices:
      print (f"\n-- connecting to {device['host']}---")
      try:
          # we can establish connection using dictionary unpacking(**)
          with ConnectHandler(**device) as net_connect: # Corrected Connecthandler to ConnectHandler
              print(f"Pushing configuration to {net_connect.find_prompt()}...")

              #Apply the configuration set
              output = net_connect.send_config_set(config_commands_multiple)
              print(output)

              #save the running configuration to NVRAM
              net_connect.send_command('write memory')
              print("Configuration saved successfully.")

      except NetmikoTimeoutException:
          print(f"Error: Connection timed out for {device['host']}.")
      except NetmikoAuthenticationException:
          print(f"Error: Authentication failed for {device['host']}.")
      except Exception as e:
          print(f"An unexpected error occurred: {e}") # Corrected string formatting and typo

if __name__ == "__main__": # Corrected _name_ to __name__
  run_automation()



-- connecting to devnetsandboxiosxec9k.cisco.com---
Pushing configuration to Cat9k_AO_Sandbox#...

Cat9k_AO_Sandbox#configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
Cat9k_AO_Sandbox(config)#interface GigabitEthernet1/0/1
Cat9k_AO_Sandbox(config-if)#no switchport
Cat9k_AO_Sandbox(config-if)#no shutdown
Cat9k_AO_Sandbox(config-if)#interface GigabitEthernet1/0/1.50
Cat9k_AO_Sandbox(config-subif)#encapsulation dot1Q 50
Cat9k_AO_Sandbox(config-subif)#ip address 192.168.50.1 255.255.255.0
Cat9k_AO_Sandbox(config-subif)#description Automation_VLAN_50
Cat9k_AO_Sandbox(config-subif)#end
Cat9k_AO_Sandbox#
Configuration saved successfully.

-- connecting to 192.0.2.10---
Error: Connection timed out for 192.0.2.10.
